<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# Shared Setup

This notebook is the common starting point for the group.

## Goals
- load the Kaggle dataset
- check class balance
- build one preprocessing flow for everyone

In [1]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so `src` is importable
repo_root = Path.cwd().resolve().parent
sys.path.insert(0, str(repo_root))

from src.config import RAW_DATA_DIR
from src.data_pipeline import (
    build_data_pipeline,
    detect_target_column,
 )

raw_dir = RAW_DATA_DIR

csv_path = raw_dir / "predictive_maintenance.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"Missing CSV at {csv_path}")

# Detect target column from the CSV header
detected_target = detect_target_column(csv_path)

artifacts = build_data_pipeline(
    str(csv_path),
    target_column=detected_target,
    test_size=0.2,
    random_state=42,
 )

artifacts.X_train.shape, artifacts.X_test.shape

((8000, 7), (2000, 7))

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import RAW_DATA_DIR
from src.preprocessing import make_shared_preprocessing_pipeline, split_data

sns.set_theme(style="whitegrid")

print("Ready to load data from:", RAW_DATA_DIR)

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
csv_files = sorted(RAW_DATA_DIR.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError('No CSV file found in data/raw. Put the Kaggle CSV there first.')

dataset_path = csv_files[0]
df = pd.read_csv(dataset_path)
print('Loaded:', dataset_path.name)
print('Shape:', df.shape)
display(df.head())

In [ ]:
print('Missing values:')
display(df.isna().sum().sort_values(ascending=False).head(10))

print('Column names:')
print(list(df.columns))

# Resolve target column with a case/whitespace-insensitive lookup
normalized = {str(col).strip().lower(): col for col in df.columns}
target_column = None
for candidate in ["machine failure", "machine_failure", "target", "label"]:
    if candidate in normalized:
        target_column = normalized[candidate]
        break

if target_column is not None:
    print('Target column:', target_column)
    print('Target balance:')
    display(df[target_column].value_counts(normalize=True).rename('ratio'))
    ax = df[target_column].value_counts().plot(kind='bar', title='Target distribution')
    ax.set_xlabel(target_column)
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No target column found. Update the candidate list or set target_column manually.')

In [ ]:
target_column = globals().get("target_column")
if not target_column:
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in ["machine failure", "machine_failure", "target", "label", "failure"]:
        if candidate in normalized:
            target_column = normalized[candidate]
            break

shared = make_shared_preprocessing_pipeline(df, target_column=target_column)
X_train, X_test, y_train, y_test = split_data(df, target=shared['target_column'])

print('Target column:', shared['target_column'])
print('Dropped columns:', shared['dropped_columns'])
print('Train shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)
print('Numeric features:', X_train.select_dtypes(include=['number']).columns.tolist())
print('Categorical features:', X_train.select_dtypes(exclude=['number']).columns.tolist())

X_train_ready = shared['preprocessor'].fit_transform(X_train)
X_test_ready = shared['preprocessor'].transform(X_test)
print('Processed train shape:', X_train_ready.shape)
print('Processed test shape:', X_test_ready.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=42,
    early_stopping=True,
 )

mlp.fit(X_train_ready, y_train)
y_pred = mlp.predict(X_test_ready)
y_proba = mlp.predict_proba(X_test_ready)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# Take one raw row, preprocess it, then predict
new_row = X_test.iloc[[0]]
new_row_ready = shared["preprocessor"].transform(new_row)

prediction = mlp.predict(new_row_ready)[0]      # 0 or 1
probability = mlp.predict_proba(new_row_ready)[0, 1]

print("Prediction:", int(prediction))
print("Failure probability:", float(probability))

In [ ]:
import json
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from src.config import RESULTS_DIR

run_id = datetime.now().strftime("shared_setup_%Y%m%d_%H%M%S")
run_dir = RESULTS_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(run_dir / "data_snapshot.csv", index=False)
X_train.to_csv(run_dir / "X_train.csv", index=False)
X_test.to_csv(run_dir / "X_test.csv", index=False)
y_train.to_csv(run_dir / "y_train.csv", index=False)
y_test.to_csv(run_dir / "y_test.csv", index=False)

np.save(run_dir / "X_train_ready.npy", X_train_ready)
np.save(run_dir / "X_test_ready.npy", X_test_ready)

feature_names = None
try:
    feature_names = shared["preprocessor"].get_feature_names_out()
    (run_dir / "feature_names.txt").write_text("\n".join(map(str, feature_names)))
except Exception:
    pass

if "target_column" in globals():
    ax = df[target_column].value_counts().plot(kind="bar", title="Target distribution")
    ax.set_xlabel(target_column)
    ax.set_ylabel("Count")
    fig = ax.get_figure()
    fig.savefig(run_dir / "target_distribution.png", bbox_inches="tight")
    plt.close(fig)

mlp_metrics = None
if "mlp" in globals() and "y_pred" in globals() and "y_proba" in globals():
    joblib.dump(mlp, run_dir / "mlp_model.joblib")
    mlp_metrics = {
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "classification_report": classification_report(y_test, y_pred, digits=3, output_dict=True),
        "params": mlp.get_params(),
    }
    (run_dir / "mlp_metrics.json").write_text(json.dumps(mlp_metrics, indent=2))

summary = {
    "run_id": run_id,
    "data_shape": list(df.shape),
    "target_column": target_column if "target_column" in globals() else None,
    "dropped_columns": shared.get("dropped_columns") if "shared" in globals() else None,
    "train_shape": [int(X_train.shape[0]), int(X_train.shape[1])],
    "test_shape": [int(X_test.shape[0]), int(X_test.shape[1])],
    "processed_train_shape": [int(X_train_ready.shape[0]), int(X_train_ready.shape[1])],
    "processed_test_shape": [int(X_test_ready.shape[0]), int(X_test_ready.shape[1])],
    "numeric_features": X_train.select_dtypes(include=["number"]).columns.tolist(),
    "categorical_features": X_train.select_dtypes(exclude=["number"]).columns.tolist(),
    "target_balance": y_train.value_counts(normalize=True).to_dict(),
    "mlp_metrics": mlp_metrics,
}
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2))

print("Saved results to:", run_dir)